# SIFT 스케일 공간 생성 (DoG Pyramid) 실습

In [ ]:
import cv2
import numpy as np
import urllib.request
from google.colab.patches import cv2_imshow

# 영상 로드 및 흑백 변환
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg'
urllib.request.urlretrieve(url, 'butterfly.jpg')
img = cv2.imread('butterfly.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 스케일 공간 설정 파라미터
num_octaves = 4 # 총 4개의 옥타브 생성
s = 3           # 하나의 옥타브 내의 인터벌 수 (SIFT 표준값)
sigma0 = 1.6
k = 2**(1.0/s)  # 스케일 배수

# 스케일 공간 생성 (Gaussian Pyramid & DoG Pyramid)
gaussian_pyr = []
dog_pyr = []

# 원본 이미지의 초기 가우시안 필터링
base_img = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma0, sigmaY=sigma0)

for o in range(num_octaves):
    current_octave_gaussians = [base_img]
    current_octave_dogs = []
    
    # 하나의 옥타브 내에서 s+3개의 가우시안 이미지를 만듦
    for i in range(1, s+3):
        # 이전 층의 sigma에 k를 곱해 가며 흐리게 만듦
        sigma = sigma0 * (k ** i)
        gaussian_img = cv2.GaussianBlur(base_img, (0, 0), sigmaX=sigma, sigmaY=sigma)
        current_octave_gaussians.append(gaussian_img)
        
        # 인접한 가우시안 이미지의 차이 (DoG) 계산 (high_sigma - low_sigma)
        # 빼기 연산 시 overflow 방지를 위해 float32 사용
        dog_img = cv2.subtract(current_octave_gaussians[i].astype(np.float32), current_octave_gaussians[i-1].astype(np.float32))
        current_octave_dogs.append(dog_img)
        
    gaussian_pyr.append(current_octave_gaussians)
    dog_pyr.append(current_octave_dogs)
    
    # 다음 옥타브를 위해 크기를 절반으로 줄임 (s번째 이미지 사용)
    base_img = cv2.resize(current_octave_gaussians[s], (0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_NEAREST)

# 결과 시각화
print(f"--- SIFT DoG Pyramid (Octaves: {num_octaves}, Intervals: {s}) ---")
for o in range(num_octaves):
    num_dogs = len(dog_pyr[o])
    viz_list = []
    for d_img in dog_pyr[o]:
        # 차이값이 매우 작으므로 잘 보이게 정규화
        norm_img = cv2.normalize(d_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        viz_list.append(norm_img)
    
    octave_viz = np.hstack(viz_list)
    print(f"\n[Octave {o}] (Size: {octave_viz.shape[1]}x{octave_viz.shape[0]})")
    cv2_imshow(octave_viz)